# Hospital Readmission Predictor using eICU-CRD v2.0

## IT3100 AI Application Project - Progress Review 1

**Objective:** Develop a machine learning model to predict hospital readmission within 30 days using the eICU Collaborative Research Database (eICU-CRD v2.0).

---

## How to Run This Notebook

### Prerequisites
- Python 3.8+
- Required packages: pandas, numpy, matplotlib, seaborn, scikit-learn, xgboost, shap
- Optional: psycopg2-binary (for direct eICU-CRD database access)

### Data Setup Options

**Option A: Use Pre-downloaded CSV Files (Recommended)**
1. Download eICU-CRD v2.0 from PhysioNet (requires credentialing): https://physionet.org/content/eicu-crd/2.0/
2. Extract relevant tables as CSV files
3. Place files in `data/raw/` folder:
   - `patient.csv`, `admissiondx.csv`, `diagnosis.csv`, `lab.csv`, `vitalperiodic.csv`, `vitalaperiodic.csv`, `medication.csv`
4. Run all cells sequentially

**Option B: Generate Synthetic Data (For Demonstration)**
- If no CSV files are found, the notebook will automatically generate synthetic data
- Synthetic data mimics eICU-CRD structure and missingness patterns
- Suitable for testing code flow and visualization outputs

### Estimated Runtime
- With synthetic data: ~2-3 minutes
- With real eICU-CRD data: ~10-15 minutes (depending on data size)

---

## Table of Contents

1. [Setup and Imports](#section-1)
2. [Data Collection and Feature Extraction](#section-2)
3. [Exploratory Data Analysis (EDA)](#section-3)
4. [Data Preparation](#section-4)
5. [Model Training and Evaluation](#section-5)
6. [Model Interpretation with SHAP](#section-6)
7. [Conclusion and Next Steps](#section-7)

<a id='section-1'></a>
## 1. Setup and Imports

This section initializes the environment by importing all necessary libraries and configuring visualization settings.

In [ ]:
"""
Import all required libraries for the Hospital Readmission Predictor.

Libraries included:
- pandas, numpy: Data manipulation
- matplotlib, seaborn: Visualization
- sklearn: Machine learning utilities
- xgboost: Gradient boosting model
- shap: Model interpretability
- os, glob: File system operations for CSV handling

Error handling: Try-except blocks ensure graceful degradation if optional packages are unavailable.
"""

import warnings
warnings.filterwarnings('ignore')

# Core data manipulation
import pandas as pd
import numpy as np

# File system operations
import os
import glob

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Configure plot aesthetics
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

# Machine learning
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    roc_curve, precision_recall_curve
)
from sklearn.impute import SimpleImputer

# XGBoost for gradient boosting
try:
    import xgboost as xgb
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False
    print("Warning: XGBoost not available. Install with: pip install xgboost")

# SHAP for model interpretation
try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print("Warning: SHAP not available. Install with: pip install shap")

# Database connection (optional, for direct eICU-CRD access)
try:
    import psycopg2
    from psycopg2 import sql
    DB_AVAILABLE = True
except ImportError:
    DB_AVAILABLE = False
    print("Note: psycopg2 not available. Will use CSV files or synthetic data.")

print("All core libraries imported successfully.")
print(f"XGBoost available: {XGB_AVAILABLE}")
print(f"SHAP available: {SHAP_AVAILABLE}")

<a id='section-2'></a>
## 2. Data Collection and Feature Extraction

### 2.1 Overview of eICU-CRD v2.0 Database Structure

The eICU-CRD v2.0 database contains de-identified health data for over 200,000 ICU stays. Key tables used in this project:

| Table | Description | Key Features |
|-------|-------------|---------------|
| `patient` | Patient demographics and stay information | age, gender, weight, height, admission/discharge offsets |
| `admissiondx` | Admission diagnoses | Primary diagnosis at ICU admission |
| `diagnosis` | All diagnoses during stay | ICD-9/10 codes, priority levels |
| `lab` | Laboratory results | HbA1c, blood tests, chemistry panels |
| `vitalperiodic` | Periodic vital signs | BP, HR, SpO2 (5-min intervals) |
| `vitalaperiodic` | Aperiodic vital signs | Spot measurements |
| `medication` | Medications administered | Drug names, doses, routes |

### 2.2 Feature Extraction Mapping

| Feature | Source Table | Extraction Logic |
|---------|--------------|------------------|
| Prior admissions | `admissiondx` | Count previous `patientunitstayid` per `patienthealthsystemstayid` |
| Comorbidity count | `diagnosis` | Elixhauser/Charlson index from ICD-9/10 codes |
| BMI | `patient` | Calculate from `admissionweight`, `dischargeweight`, `height` |
| HbA1c | `lab` | Extract where `labname` = 'HbA1c' (expect ~5-10% coverage) |
| Systolic BP | `vitalperiodic`/`vitalaperiodic` | Mean/median of non-null values |
| Medication count | `medication` | Count distinct entries per stay |
| Age | `patient` | Categorize: <30, 30–59, 60–89, >90 |
| Discharge diagnosis | `diagnosis` | Where `priority` = 'Primary' |

### 2.3 Target Variable Definition

**Readmission within 30 days:** Binary variable indicating whether a patient has a subsequent `patienthealthsystemstayid` with `admitoffset` within 30 days (4320 minutes) of current `dischargeoffset`.

### 2.4 Data Directory Setup

This cell creates the required directory structure for storing raw and processed data files.

In [ ]:
"""
Set up data directory structure following best practices:
- data/raw/: Stores original CSV files (patient.csv, diagnosis.csv, etc.)
- data/processed/: Stores cleaned and merged datasets

The os.makedirs() function with exist_ok=True ensures directories are created only if they don't exist.
"""

# Define data directories
DATA_RAW_DIR = 'data/raw'
DATA_PROCESSED_DIR = 'data/processed'

# Create directories if they don't exist
os.makedirs(DATA_RAW_DIR, exist_ok=True)
os.makedirs(DATA_PROCESSED_DIR, exist_ok=True)

print(f"Data directories created/verified:")
print(f"  - Raw data: {DATA_RAW_DIR}/")
print(f"  - Processed data: {DATA_PROCESSED_DIR}/")

### 2.5 Load Data from CSV Files

This function loads eICU-CRD tables from CSV files stored in the `data/raw/` directory. If files are not found, synthetic data will be generated.

In [ ]:
"""
Load eICU-CRD data from CSV files.

Expected files in data/raw/:
- patient.csv: Patient demographics and stay information
- admissiondx.csv: Admission diagnoses
- diagnosis.csv: All diagnoses with ICD codes
- lab.csv: Laboratory results including HbA1c
- vitalperiodic.csv: Periodic vital signs
- vitalaperiodic.csv: Aperiodic vital signs
- medication.csv: Medication records

Error handling: Returns None if any critical file is missing, triggering synthetic data generation.
"""

def load_csv_data(raw_dir='data/raw'):
    """
    Load all required eICU-CRD tables from CSV files.
    
    Args:
        raw_dir (str): Path to raw data directory
    
    Returns:
        dict: Dictionary of DataFrames keyed by table name, or None if files missing
    """
    required_files = [
        'patient.csv',
        'admissiondx.csv',
        'diagnosis.csv',
        'lab.csv',
        'vitalperiodic.csv',
        'vitalaperiodic.csv',
        'medication.csv'
    ]
    
    dataframes = {}
    missing_files = []
    
    for filename in required_files:
        filepath = os.path.join(raw_dir, filename)
        table_name = filename.replace('.csv', '')
        
        if os.path.exists(filepath):
            try:
                print(f"Loading {filename}...")
                df = pd.read_csv(filepath)
                dataframes[table_name] = df
                print(f"  - Loaded {len(df)} rows, {len(df.columns)} columns")
            except Exception as e:
                print(f"  - Error reading {filename}: {e}")
                missing_files.append(filename)
        else:
            print(f"  - File not found: {filepath}")
            missing_files.append(filename)
    
    if missing_files:
        print(f"\nMissing {len(missing_files)} required files: {missing_files}")
        print("Will proceed with synthetic data generation.")
        return None
    
    print(f"\nSuccessfully loaded all {len(required_files)} CSV files.")
    return dataframes

# Attempt to load CSV data
csv_data = load_csv_data(DATA_RAW_DIR)

### 2.6 Synthetic Data Generation (Fallback)

If CSV files are not available, this section generates realistic synthetic data that mimics eICU-CRD structure and characteristics, including expected missingness patterns (e.g., ~90% missing HbA1c).

In [ ]:
"""
Generate synthetic eICU-CRD-like data for demonstration purposes.

This function creates realistic synthetic data with:
- Proper data types and value ranges matching eICU-CRD
- Expected missingness patterns (e.g., HbA1c ~90% missing)
- Realistic correlations between features
- Target variable with ~15-20% readmission rate (matching literature)

Note: This is for code testing and visualization demonstration only.
Real analysis should use actual eICU-CRD data.
"""

def generate_synthetic_eicu_data(n_samples=5000, seed=42):
    """
    Generate synthetic dataset mimicking eICU-CRD v2.0 structure.
    
    Args:
        n_samples (int): Number of patient stays to generate
        seed (int): Random seed for reproducibility
    
    Returns:
        pd.DataFrame: Synthetic dataset with all required features
    """
    np.random.seed(seed)
    
    print(f"Generating synthetic data for {n_samples} patient stays...")
    
    # Generate unique IDs
    patient_ids = np.arange(1, n_samples + 1)
    healthsystem_ids = np.random.choice(np.arange(1, n_samples // 2 + 1), n_samples)
    
    # Age: Skewed toward older patients (ICU demographic)
    age = np.clip(np.random.normal(65, 18, n_samples), 18, 100).astype(int)
    
    # Gender: Roughly balanced
    gender = np.random.choice(['Male', 'Female'], n_samples)
    
    # Weight and Height for BMI calculation
    admission_weight = np.clip(np.random.normal(80, 20, n_samples), 30, 200)
    height = np.clip(np.random.normal(170, 10, n_samples), 140, 200)
    
    # Calculate BMI (kg/m^2)
    bmi = admission_weight / ((height / 100) ** 2)
    
    # Prior admissions: Most patients have 0-2 prior admissions
    prior_admissions = np.random.poisson(1, n_samples)
    
    # Comorbidity count: Correlated with age
    comorbidity_count = np.clip(np.random.poisson(2 + age / 30, n_samples), 0, 15)
    
    # HbA1c: ~90% missingness (as expected in eICU-CRD)
    hba1c = np.full(n_samples, np.nan)
    hba1c_available_idx = np.random.choice(n_samples, size=int(n_samples * 0.1), replace=False)
    hba1c[hba1c_available_idx] = np.clip(np.random.normal(6.5, 1.5, len(hba1c_available_idx)), 4, 14)
    
    # Systolic BP: Normal range with some outliers
    systolic_bp = np.clip(np.random.normal(125, 20, n_samples), 70, 200)
    
    # Medication count: Correlated with comorbidity
    medication_count = np.clip(np.random.poisson(3 + comorbidity_count, n_samples), 0, 20)
    
    # Primary diagnosis categories
    diagnosis_categories = ['Cardiac', 'Respiratory', 'Neurological', 'Sepsis', 'Trauma', 'Other']
    primary_diagnosis = np.random.choice(
        diagnosis_categories, 
        n_samples, 
        p=[0.25, 0.20, 0.15, 0.20, 0.10, 0.10]
    )
    
    # Target variable: 30-day readmission (~18% rate, correlated with features)
    readmission_prob = (
        0.10 +
        0.02 * (prior_admissions > 1) +
        0.03 * (comorbidity_count > 3) +
        0.02 * (age > 70) +
        0.01 * (~np.isnan(hba1c)) * (hba1c > 7) +
        np.random.normal(0, 0.05, n_samples)
    )
    readmission_prob = np.clip(readmission_prob, 0, 1)
    readmitted_30day = (np.random.random(n_samples) < readmission_prob).astype(int)
    
    # Create DataFrame
    df = pd.DataFrame({
        'patientunitstayid': patient_ids,
        'patienthealthsystemstayid': healthsystem_ids,
        'age': age,
        'gender': gender,
        'admissionweight': admission_weight,
        'height': height,
        'bmi': bmi,
        'prior_admissions': prior_admissions,
        'comorbidity_count': comorbidity_count,
        'hba1c': hba1c,
        'mean_systolic_bp': systolic_bp,
        'medication_count': medication_count,
        'primary_diagnosis': primary_diagnosis,
        'readmitted_30day': readmitted_30day
    })
    
    print(f"Synthetic data generated: {df.shape[0]} rows, {df.shape[1]} columns")
    print(f"Readmission rate: {df['readmitted_30day'].mean():.1%}")
    print(f"HbA1c availability: {(~df['hba1c'].isna()).sum() / len(df):.1%}")
    
    return df

# Generate or load data
if csv_data is not None:
    # Would merge CSV data here (simplified for this example)
    print("CSV data loaded - would proceed with feature extraction.")
    df = pd.DataFrame()  # Placeholder
else:
    df = generate_synthetic_eicu_data(n_samples=5000)
    
    # Save synthetic data for reference
    df.to_csv(os.path.join(DATA_PROCESSED_DIR, 'synthetic_features.csv'), index=False)
    print(f"\nSynthetic data saved to {DATA_PROCESSED_DIR}/synthetic_features.csv")

### 2.7 Initial Data Overview

Display basic information about the loaded dataset including shape, data types, and sample records.

In [ ]:
"""
Display initial dataset overview.

This provides a quick sanity check of the data structure before proceeding with EDA.
We examine:
- Dataset dimensions (rows, columns)
- Data types of each column
- First few records
- Basic statistics
"""

print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)

print(f"\nDataset Shape: {df.shape[0]} rows × {df.shape[1]} columns")

print("\nColumn Data Types:")
print(df.dtypes)

print("\nFirst 5 Records:")
display(df.head())

print("\nBasic Statistics:")
display(df.describe())

<a id='section-3'></a>
## 3. Exploratory Data Analysis (EDA)

This section performs comprehensive exploratory data analysis with visualizations to understand:
- Missing data patterns (especially for HbA1c)
- Target variable distribution and class imbalance
- Feature distributions and outliers
- Correlations between features
- Relationships between features and readmission

### 3.1 Missing Data Analysis

Visualize missing data patterns across all features. HbA1c is expected to have high missingness (~90%).

In [ ]:
"""
Analyze and visualize missing data patterns.

Key objectives:
1. Quantify missingness percentage for each feature
2. Visualize missing data matrix to identify patterns
3. Confirm expected HbA1c missingness (~90%)

Missing data handling strategy will be determined based on these findings.
"""

# Calculate missing data statistics
missing_stats = pd.DataFrame({
    'Missing Count': df.isna().sum(),
    'Missing Percentage': (df.isna().sum() / len(df) * 100).round(2),
    'Non-Missing Count': df.notna().sum(),
    'Non-Missing Percentage': (df.notna().sum() / len(df) * 100).round(2)
})

missing_stats = missing_stats[missing_stats['Missing Count'] > 0].sort_values('Missing Percentage', ascending=False)

print("Missing Data Summary:")
print(missing_stats)

# Create figure with subplots
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Missing data percentage bar chart
if len(missing_stats) > 0:
    sns.barplot(
        x=missing_stats.index, 
        y=missing_stats['Missing Percentage'], 
        ax=axes[0],
        palette='Reds_r'
    )
    axes[0].set_title('Missing Data Percentage by Feature', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Feature', fontsize=12)
    axes[0].set_ylabel('Missing Percentage (%)', fontsize=12)
    axes[0].tick_params(axis='x', rotation=45)
    
    # Add value labels
    for i, v in enumerate(missing_stats['Missing Percentage']):
        axes[0].text(i, v + 0.5, f'{v:.1f}%', ha='center', va='bottom', fontsize=10)
else:
    axes[0].text(0.5, 0.5, 'No missing data', ha='center', va='center', transform=axes[0].transAxes)
    axes[0].set_title('Missing Data Percentage', fontsize=14)

# Plot 2: Missing data heatmap (sample for large datasets)
sample_size = min(500, len(df))
df_sample = df.sample(sample_size, random_state=42).reset_index(drop=True)
missing_matrix = df_sample.isna()

# Create binary mask for visualization
sns.heatmap(
    missing_matrix.T,
    cmap='binary',
    cbar_kws={'label': 'Missing (Black) / Present (White)'},
    ax=axes[1]
)
axes[1].set_title(f'Missing Data Pattern (Sample of {sample_size} records)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Record Index', fontsize=12)
axes[1].set_ylabel('Feature', fontsize=12)

plt.tight_layout()
plt.savefig(os.path.join(DATA_PROCESSED_DIR, 'missing_data_analysis.png'), dpi=150, bbox_inches='tight')
print(f"\nVisualization saved to {DATA_PROCESSED_DIR}/missing_data_analysis.png")
plt.show()

# Key insight
if 'hba1c' in missing_stats.index:
    hba1c_missing = missing_stats.loc['hba1c', 'Missing Percentage']
    print(f"\n*** KEY FINDING: HbA1c is {hba1c_missing}% missing, confirming expectation of high missingness. ***")
    print("This will require imputation or exclusion in data preparation.")

### 3.2 Target Variable Distribution

Analyze the distribution of the target variable (30-day readmission) to assess class imbalance.

In [ ]:
"""
Analyze target variable distribution.

Key objectives:
1. Calculate class frequencies and percentages
2. Compute imbalance ratio
3. Visualize distribution with bar chart

Class imbalance will inform our modeling strategy (e.g., class weights, SMOTE).
"""

# Calculate target statistics
target_counts = df['readmitted_30day'].value_counts()
target_pct = df['readmitted_30day'].value_counts(normalize=True) * 100

readmission_rate = (df['readmitted_30day'] == 1).sum() / len(df)
imbalance_ratio = (df['readmitted_30day'] == 0).sum() / (df['readmitted_30day'] == 1).sum()

print("Target Variable Distribution (30-Day Readmission):")
print("=" * 50)
print(f"\nClass Counts:")
print(f"  Not Readmitted (0): {target_counts.get(0, 0):,} ({target_pct.get(0, 0):.1f}%)")
print(f"  Readmitted (1):     {target_counts.get(1, 0):,} ({target_pct.get(1, 0):.1f}%)")
print(f"\nReadmission Rate: {readmission_rate:.1%}")
print(f"Imbalance Ratio (Majority:Minority): {imbalance_ratio:.1f}:1")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart with counts
bars = axes[0].bar(
    ['Not Readmitted', 'Readmitted'],
    [target_counts.get(0, 0), target_counts.get(1, 0)],
    color=['#2ecc71', '#e74c3c'],
    edgecolor='black',
    linewidth=1.5
)
axes[0].set_title('Target Variable Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_xlabel('Readmission Status', fontsize=12)

# Add value labels on bars
for bar, count in zip(bars, [target_counts.get(0, 0), target_counts.get(1, 0)]):
    height = bar.get_height()
    axes[0].text(
        bar.get_x() + bar.get_width() / 2.,
        height + height * 0.02,
        f'{count:,}\n({target_pct.get(1 if bar.get_x() > 0 else 0, 0):.1f}%)',
        ha='center',
        va='bottom',
        fontsize=12,
        fontweight='bold'
    )

# Pie chart
colors = ['#2ecc71', '#e74c3c']
axes[1].pie(
    [target_pct.get(0, 0), target_pct.get(1, 0)],
    labels=['Not Readmitted', 'Readmitted'],
    autopct='%1.1f%%',
    colors=colors,
    startangle=90,
    explode=(0.05, 0.05)
)
axes[1].set_title('Readmission Proportion', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(DATA_PROCESSED_DIR, 'target_distribution.png'), dpi=150, bbox_inches='tight')
print(f"\nVisualization saved to {DATA_PROCESSED_DIR}/target_distribution.png")
plt.show()

# Imbalance assessment
if imbalance_ratio > 3:
    print(f"\n*** WARNING: Significant class imbalance detected ({imbalance_ratio:.1f}:1). ***")
    print("Will apply class weighting or resampling techniques during modeling.")
elif imbalance_ratio > 2:
    print(f"\n*** NOTE: Moderate class imbalance ({imbalance_ratio:.1f}:1). ***")
    print("Class weighting recommended for optimal model performance.")
else:
    print(f"\n*** Class distribution is reasonably balanced ({imbalance_ratio:.1f}:1). ***")

### 3.3 Feature Distributions

Examine the distribution of key numerical features to identify skewness, outliers, and data quality issues.

In [ ]:
"""
Visualize distributions of key numerical features.

Features analyzed:
- Age: Check demographic distribution
- BMI: Identify underweight/obese populations
- Comorbidity count: Assess disease burden
- Systolic BP: Check vital sign distribution
- Medication count: Evaluate polypharmacy patterns

KDE plots overlaid on histograms provide smooth density estimates.
"""

numerical_features = ['age', 'bmi', 'comorbidity_count', 'mean_systolic_bp', 'medication_count', 'prior_admissions']

# Filter out features not in dataframe
numerical_features = [f for f in numerical_features if f in df.columns]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, feature in enumerate(numerical_features):
    # Create histogram with KDE
    sns.histplot(
        data=df,
        x=feature,
        kde=True,
        ax=axes[idx],
        color='steelblue',
        edgecolor='black',
        alpha=0.7,
        bins=30
    )
    
    # Add statistics
    mean_val = df[feature].mean()
    median_val = df[feature].median()
    std_val = df[feature].std()
    
    axes[idx].axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_val:.1f}')
    axes[idx].axvline(median_val, color='green', linestyle='-', linewidth=2, label=f'Median: {median_val:.1f}')
    
    axes[idx].set_title(f'{feature.replace("_", " ").title()} Distribution', fontsize=13, fontweight='bold')
    axes[idx].set_xlabel(feature.replace('_', ' ').title(), fontsize=11)
    axes[idx].set_ylabel('Frequency', fontsize=11)
    axes[idx].legend(loc='best', fontsize=9)
    axes[idx].grid(True, alpha=0.3)

# Hide empty subplot if fewer than 6 features
if len(numerical_features) < 6:
    for i in range(len(numerical_features), 6):
        axes[i].axis('off')

plt.suptitle('Numerical Feature Distributions', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(DATA_PROCESSED_DIR, 'feature_distributions.png'), dpi=150, bbox_inches='tight')
print(f"Visualization saved to {DATA_PROCESSED_DIR}/feature_distributions.png")
plt.show()

# Print summary statistics
print("\nFeature Distribution Summary:")
print("=" * 70)
summary_df = df[numerical_features].describe().loc[['mean', 'std', 'min', '25%', '50%', '75%', 'max']]
display(summary_df.round(2))

### 3.4 Age Category Analysis

Analyze readmission rates across different age categories as defined in the project requirements (<30, 30-59, 60-89, >90).

In [ ]:
"""
Analyze readmission rates by age category.

Age categories (as per project requirements):
- <30 years: Young adults
- 30-59 years: Middle-aged adults
- 60-89 years: Older adults
- >90 years: Very elderly

This analysis helps identify age-related risk patterns for readmission.
"""

# Create age categories
def categorize_age(age):
    if age < 30:
        return '<30'
    elif age < 60:
        return '30-59'
    elif age <= 89:
        return '60-89'
    else:
        return '>90'

df['age_category'] = df['age'].apply(categorize_age)

# Calculate readmission rates by age category
age_readmission = df.groupby('age_category').agg({
    'readmitted_30day': ['count', 'sum', 'mean']
}).round(4)

age_readmission.columns = ['Total Patients', 'Readmitted', 'Readmission Rate']
age_readmission['Readmission Rate (%)'] = (age_readmission['Readmission Rate'] * 100).round(2)

# Reorder categories
age_order = ['<30', '30-59', '60-89', '>90']
age_readmission = age_readmission.reindex([cat for cat in age_order if cat in age_readmission.index])

print("Readmission Rates by Age Category:")
print("=" * 60)
display(age_readmission)

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart: Readmission rate by age
if len(age_readmission) > 0:
    bars = axes[0].bar(
        age_readmission.index,
        age_readmission['Readmission Rate (%)'],
        color=sns.color_palette('Blues_d', len(age_readmission)),
        edgecolor='black',
        linewidth=1.5
    )
    
    axes[0].set_title('Readmission Rate by Age Category', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Age Category (years)', fontsize=12)
    axes[0].set_ylabel('Readmission Rate (%)', fontsize=12)
    axes[0].set_ylim(0, max(age_readmission['Readmission Rate (%)']) * 1.2)
    
    # Add value labels
    for bar, rate in zip(bars, age_readmission['Readmission Rate (%)']):
        height = bar.get_height()
        axes[0].text(
            bar.get_x() + bar.get_width() / 2.,
            height + 0.5,
            f'{rate:.1f}%',
            ha='center',
            va='bottom',
            fontsize=11,
            fontweight='bold'
        )

# Stacked bar: Absolute counts
if len(age_readmission) > 0:
    age_readmission.plot(
        kind='bar',
        stacked=True,
        ax=axes[1],
        color=['#2ecc71', '#e74c3c'],
        edgecolor='black'
    )
    axes[1].set_title('Patient Counts by Age and Readmission Status', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Age Category', fontsize=12)
    axes[1].set_ylabel('Number of Patients', fontsize=12)
    axes[1].legend(['Not Readmitted', 'Readmitted'], loc='upper left')
    axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig(os.path.join(DATA_PROCESSED_DIR, 'age_category_analysis.png'), dpi=150, bbox_inches='tight')
print(f"\nVisualization saved to {DATA_PROCESSED_DIR}/age_category_analysis.png")
plt.show()

# Key finding
if len(age_readmission) > 0:
    highest_risk = age_readmission['Readmission Rate'].idxmax()
    lowest_risk = age_readmission['Readmission Rate'].idxmin()
    print(f"\n*** KEY FINDING: Highest readmission rate in '{highest_risk}' age group ({age_readmission.loc[highest_risk, 'Readmission Rate (%)']:.1f}%). ***")
    print(f"*** Lowest readmission rate in '{lowest_risk}' age group ({age_readmission.loc[lowest_risk, 'Readmission Rate (%)']:.1f}%). ***")

### 3.5 Correlation Analysis

Examine correlations between numerical features and with the target variable using a heatmap with hierarchical clustering.

In [ ]:
"""
Generate correlation heatmap with hierarchical clustering.

Objectives:
1. Identify highly correlated features (multicollinearity concerns)
2. Find features most strongly associated with readmission
3. Visualize correlation structure using clustered heatmap

Hierarchical clustering groups similar features together for easier pattern recognition.
"""

# Select numerical features for correlation
corr_features = ['age', 'bmi', 'prior_admissions', 'comorbidity_count', 
                 'mean_systolic_bp', 'medication_count', 'readmitted_30day']
corr_features = [f for f in corr_features if f in df.columns]

# Calculate correlation matrix
corr_matrix = df[corr_features].corr(method='pearson')

print("Correlation Matrix (Pearson):")
print("=" * 60)
display(corr_matrix.round(3))

# Visualization with clustering
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Standard heatmap
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={'label': 'Correlation Coefficient'},
    ax=axes[0],
    annot_kws={'fontsize': 10}
)
axes[0].set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')
axes[0].set_yticklabels(axes[0].get_yticklabels(), rotation=0)

# Plot 2: Clustered heatmap
sns.clustermap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={'label': 'Correlation Coefficient'},
    figsize=(10, 10),
    ax=None  # Will create new figure
)

# Remove the clustermap from current axes and create separately
plt.close()  # Close the auto-generated clustermap figure

# Create proper clustermap in separate figure
g = sns.clustermap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={'label': 'Correlation Coefficient'},
    figsize=(10, 10)
)
g.fig.suptitle('Clustered Correlation Heatmap', fontsize=14, fontweight='bold', y=1.02)
plt.savefig(os.path.join(DATA_PROCESSED_DIR, 'correlation_heatmap.png'), dpi=150, bbox_inches='tight')
print(f"\nVisualizations saved to {DATA_PROCESSED_DIR}/correlation_heatmap.png")
plt.show()

# Display both plots
plt.tight_layout()
plt.show()

# Key correlations with target
if 'readmitted_30day' in corr_matrix.columns:
    target_corrs = corr_matrix['readmitted_30day'].drop('readmitted_30day').abs().sort_values(ascending=False)
    print("\n*** Top Correlations with Readmission: ***")
    for feature, corr in target_corrs.head(3).items():
        actual_corr = corr_matrix.loc[feature, 'readmitted_30day']
        print(f"  - {feature}: r = {actual_corr:.3f}")

### 3.6 Primary Diagnosis Distribution

Analyze the distribution of primary diagnoses and their association with readmission rates.

In [ ]:
"""
Analyze primary diagnosis distribution and readmission rates.

Objectives:
1. Show frequency of each diagnosis category
2. Calculate readmission rate per diagnosis
3. Identify high-risk diagnosis categories

This helps understand which conditions are most associated with readmission.
"""

if 'primary_diagnosis' in df.columns:
    # Calculate diagnosis statistics
    diag_stats = df.groupby('primary_diagnosis').agg({
        'readmitted_30day': ['count', 'sum', 'mean']
    }).round(4)
    
    diag_stats.columns = ['Total Patients', 'Readmitted', 'Readmission Rate']
    diag_stats['Readmission Rate (%)'] = (diag_stats['Readmission Rate'] * 100).round(2)
    diag_stats = diag_stats.sort_values('Total Patients', ascending=False)
    
    print("Primary Diagnosis Distribution and Readmission Rates:")
    print("=" * 60)
    display(diag_stats)
    
    # Visualization
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Bar chart: Diagnosis frequency
    sns.barplot(
        x=diag_stats.index,
        y=diag_stats['Total Patients'],
        ax=axes[0],
        palette='viridis',
        edgecolor='black'
    )
    axes[0].set_title('Patient Count by Primary Diagnosis', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Diagnosis Category', fontsize=12)
    axes[0].set_ylabel('Number of Patients', fontsize=12)
    axes[0].tick_params(axis='x', rotation=45)
    
    # Add value labels
    for i, v in enumerate(diag_stats['Total Patients']):
        axes[0].text(i, v + max(diag_stats['Total Patients']) * 0.02, f'{v:,}', ha='center', va='bottom', fontsize=10)
    
    # Bar chart: Readmission rate by diagnosis
    sns.barplot(
        x=diag_stats.index,
        y=diag_stats['Readmission Rate (%)'],
        ax=axes[1],
        palette='Reds',
        edgecolor='black'
    )
    axes[1].set_title('Readmission Rate by Primary Diagnosis', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Diagnosis Category', fontsize=12)
    axes[1].set_ylabel('Readmission Rate (%)', fontsize=12)
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].axhline(df['readmitted_30day'].mean() * 100, color='blue', linestyle='--', linewidth=2, label='Overall Rate')
    axes[1].legend(loc='upper right')
    
    # Add value labels
    for i, v in enumerate(diag_stats['Readmission Rate (%)']):
        axes[1].text(i, v + 0.5, f'{v:.1f}%', ha='center', va='bottom', fontsize=10)
    
    plt.tight_layout()
    plt.savefig(os.path.join(DATA_PROCESSED_DIR, 'diagnosis_analysis.png'), dpi=150, bbox_inches='tight')
    print(f"\nVisualization saved to {DATA_PROCESSED_DIR}/diagnosis_analysis.png")
    plt.show()
    
    # Key finding
    highest_risk_diag = diag_stats['Readmission Rate'].idxmax()
    print(f"\n*** KEY FINDING: '{highest_risk_diag}' has highest readmission rate ({diag_stats.loc[highest_risk_diag, 'Readmission Rate (%)']:.1f}%). ***")
else:
    print("Primary diagnosis column not found. Skipping analysis.")

<a id='section-4'></a>
## 4. Data Preparation

This section prepares the data for machine learning modeling:
1. Handle missing HbA1c values (imputation + indicator)
2. Encode categorical variables
3. Scale numerical features
4. Split into training and testing sets
5. Address class imbalance

### 4.1 Handling Missing HbA1c Values

Strategy: Create a binary indicator for HbA1c availability AND impute missing values with median. This preserves the information that HbA1c was not measured (which may be clinically meaningful).

In [ ]:
"""
Handle missing HbA1c values using a two-pronged approach:

1. Create binary indicator variable (hba1c_available): 
   - 1 if HbA1c was measured, 0 otherwise
   - Preserves information about test ordering patterns

2. Impute missing HbA1c values with median:
   - Uses median of available values (robust to outliers)
   - Allows inclusion of HbA1c in models without dropping rows

Rationale: HbA1c missingness is not random - it reflects clinical decisions.
The indicator captures this pattern while imputation enables modeling.
"""

print("Handling HbA1c Missing Values")
print("=" * 50)

# Store original missingness info
hba1c_missing_before = df['hba1c'].isna().sum()
hba1c_total = len(df)

# Step 1: Create indicator variable
df['hba1c_available'] = (~df['hba1c'].isna()).astype(int)

# Step 2: Calculate median of available values
hba1c_median = df['hba1c'].median()
print(f"HbA1c median (available values): {hba1c_median:.2f}")

# Step 3: Impute missing values
df['hba1c_imputed'] = df['hba1c'].fillna(hba1c_median)

# Verify imputation
hba1c_missing_after = df['hba1c_imputed'].isna().sum()

print(f"\nBefore imputation: {hba1c_missing_before:,} missing ({hba1c_missing_before/hba1c_total*100:.1f}%)")
print(f"After imputation: {hba1c_missing_after:,} missing ({hba1c_missing_after/hba1c_total*100:.1f}%)")
print(f"Indicator variable created: {df['hba1c_available'].sum()} patients with HbA1c measured")

# Drop original hba1c column (keep imputed version and indicator)
df = df.drop(columns=['hba1c'])

print("\nHbA1c handling complete.")
print("Columns added: 'hba1c_available' (indicator), 'hba1c_imputed' (imputed values)")
print("Column removed: 'hba1c' (original with missing values)")

### 4.2 Categorical Variable Encoding

Encode categorical features using appropriate strategies:
- Binary encoding for gender (Male/Female)
- Ordinal encoding for age categories
- One-hot encoding for primary diagnosis

In [ ]:
"""
Encode categorical variables using appropriate strategies:

1. Gender (Binary): Male=0, Female=1
   - Simple binary encoding for two categories

2. Age Category (Ordinal): <30=0, 30-59=1, 60-89=2, >90=3
   - Ordinal encoding preserves age ordering

3. Primary Diagnosis (One-Hot): Separate binary column for each category
   - No inherent ordering, so one-hot encoding is appropriate
   - Creates dummy variables to avoid multicollinearity
"""

print("Encoding Categorical Variables")
print("=" * 50)

# Initialize encoder
label_encoder = LabelEncoder()

# 1. Encode Gender (Binary)
if 'gender' in df.columns:
    df['gender_encoded'] = label_encoder.fit_transform(df['gender'])
    # Map: Female=0, Male=1 (alphabetical order)
    print(f"Gender encoded: {dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))}")

# 2. Encode Age Category (Ordinal)
if 'age_category' in df.columns:
    age_mapping = {'<30': 0, '30-59': 1, '60-89': 2, '>90': 3}
    df['age_category_encoded'] = df['age_category'].map(age_mapping)
    print(f"Age category mapped: {age_mapping}")

# 3. One-Hot Encode Primary Diagnosis
if 'primary_diagnosis' in df.columns:
    diagnosis_dummies = pd.get_dummies(df['primary_diagnosis'], prefix='diagnosis', drop_first=False)
    df = pd.concat([df, diagnosis_dummies], axis=1)
    print(f"Primary diagnosis one-hot encoded: {list(diagnosis_dummies.columns)}")

print("\nEncoding complete.")
print(f"DataFrame now has {df.shape[1]} columns after encoding.")

### 4.3 Feature Selection and Final Dataset

Select final features for modeling and prepare the clean dataset.

In [ ]:
"""
Select final features for modeling.

Features included:
- Demographics: age, age_category_encoded, gender_encoded
- Clinical: bmi, prior_admissions, comorbidity_count, medication_count
- Vitals: mean_systolic_bp
- Labs: hba1c_imputed, hba1c_available
- Diagnoses: One-hot encoded diagnosis columns

Features excluded:
- IDs: patientunitstayid, patienthealthsystemstayid (not predictive)
- Original categorical columns (replaced by encoded versions)
- Raw weight/height (BMI already calculated)
"""

# Define feature columns for modeling
feature_columns = [
    'age',
    'age_category_encoded',
    'gender_encoded',
    'bmi',
    'prior_admissions',
    'comorbidity_count',
    'medication_count',
    'mean_systolic_bp',
    'hba1c_imputed',
    'hba1c_available'
]

# Add one-hot encoded diagnosis columns if they exist
diagnosis_cols = [col for col in df.columns if col.startswith('diagnosis_')]
feature_columns.extend(diagnosis_cols)

# Filter to only include columns that exist in dataframe
feature_columns = [col for col in feature_columns if col in df.columns]

# Create feature matrix X and target vector y
X = df[feature_columns].copy()
y = df['readmitted_30day'].copy()

print("Final Feature Set for Modeling:")
print("=" * 50)
print(f"Number of features: {len(feature_columns)}")
print(f"Feature names: {feature_columns}")
print(f"\nFeature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")

# Save processed dataset
processed_df = df[feature_columns + ['readmitted_30day']].copy()
processed_df.to_csv(os.path.join(DATA_PROCESSED_DIR, 'final_dataset.csv'), index=False)
print(f"\nProcessed dataset saved to {DATA_PROCESSED_DIR}/final_dataset.csv")

### 4.4 Feature Scaling

Apply StandardScaler to normalize numerical features to zero mean and unit variance. This is important for logistic regression and improves XGBoost convergence.

In [ ]:
"""
Apply feature scaling using StandardScaler.

StandardScaler transforms features to have:
- Mean = 0
- Standard deviation = 1

Benefits:
- Improves convergence for gradient-based algorithms
- Ensures all features contribute equally to distance calculations
- Required for proper regularization in logistic regression

Note: Scaler is fit on training data only to prevent data leakage.
Test data is transformed using training scaler parameters.
"""

# Initialize scaler
scaler = StandardScaler()

# Note: We'll fit the scaler after train-test split to prevent data leakage
# For now, just confirm scaler is ready
print("StandardScaler initialized and ready.")
print("\nScaling will be applied after train-test split to prevent data leakage.")
print("Scaler will be fit on training data only, then applied to both train and test sets.")

### 4.5 Train-Test Split

Split data into training (80%) and testing (20%) sets using stratified sampling to maintain class distribution.

In [ ]:
"""
Split data into training and testing sets.

Parameters:
- test_size=0.2: 80% training, 20% testing
- stratify=y: Maintain class distribution in both sets
- random_state=42: Ensure reproducibility

Stratified splitting is crucial for imbalanced datasets to ensure
both sets have similar proportions of readmitted/non-readmitted patients.
"""

# Perform stratified train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train-Test Split Results:")
print("=" * 50)
print(f"Training set: {X_train.shape[0]} samples")
print(f"Testing set:  {X_test.shape[0]} samples")
print(f"\nTraining class distribution:")
print(f"  Not readmitted: {(y_train == 0).sum()} ({(y_train == 0).mean()*100:.1f}%)")
print(f"  Readmitted:     {(y_train == 1).sum()} ({(y_train == 1).mean()*100:.1f}%)")
print(f"\nTesting class distribution:")
print(f"  Not readmitted: {(y_test == 0).sum()} ({(y_test == 0).mean()*100:.1f}%)")
print(f"  Readmitted:     {(y_test == 1).sum()} ({(y_test == 1).mean()*100:.1f}%)")

# Apply feature scaling
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nFeature scaling applied.")
print(f"Training features scaled: mean={X_train_scaled.mean():.6f}, std={X_train_scaled.std():.6f}")
print(f"Testing features scaled using training parameters.")

# Convert back to DataFrame for easier interpretation
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X.columns, index=X_train.index)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X.columns, index=X_test.index)

### 4.6 Class Imbalance Handling

Calculate class weights to address imbalance during model training. Both Logistic Regression and XGBoost support class weighting.

In [ ]:
"""
Calculate class weights for handling imbalanced data.

Strategy: Inverse frequency weighting
- Minority class (readmitted) gets higher weight
- Majority class (not readmitted) gets lower weight

This makes the model pay more attention to correctly classifying
the minority class, which is often more important in healthcare
(missing a readmission risk is costlier than false alarm).

Both Logistic Regression and XGBoost support class weights:
- LogisticRegression: class_weight='balanced' or custom dict
- XGBoost: scale_pos_weight parameter
"""

from sklearn.utils.class_weight import compute_class_weight

# Compute class weights
classes = np.unique(y_train)
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)

class_weight_dict = dict(zip(classes, class_weights))

print("Class Weights for Imbalanced Data:")
print("=" * 50)
print(f"Class 0 (Not Readmitted): weight = {class_weight_dict[0]:.3f}")
print(f"Class 1 (Readmitted):     weight = {class_weight_dict[1]:.3f}")
print(f"\nWeight ratio (Minority/Majority): {class_weight_dict[1]/class_weight_dict[0]:.2f}x")

# Calculate scale_pos_weight for XGBoost
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos_weight = neg_count / pos_count

print(f"\nXGBoost scale_pos_weight: {scale_pos_weight:.3f}")
print("\nThese weights will be used during model training.")

<a id='section-5'></a>
## 5. Model Training and Evaluation

This section trains and evaluates two supervised learning models:
1. **Logistic Regression** - Baseline linear model
2. **XGBoost** - Advanced gradient boosting model

Both models use hyperparameter tuning via GridSearchCV and are evaluated using multiple metrics appropriate for imbalanced classification.

### 5.1 Model 1: Logistic Regression (Baseline)

Train a Logistic Regression model with hyperparameter tuning. This serves as our baseline for comparison.

In [ ]:
"""
Train Logistic Regression model with hyperparameter tuning.

Hyperparameters tuned:
- C: Inverse regularization strength (smaller = stronger regularization)
- solver: Optimization algorithm ('liblinear' good for small datasets, 'lbfgs' for larger)
- penalty: Regularization type ('l1' or 'l2')

Class weighting is applied to handle imbalanced data.
Cross-validation (5-fold) ensures robust hyperparameter selection.
"""

print("Training Logistic Regression Model")
print("=" * 60)

# Define parameter grid for GridSearchCV
lr_param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'solver': ['liblinear', 'lbfgs'],
    'penalty': ['l2']  # l1 only works with liblinear
}

# Initialize Logistic Regression with class weights
lr_base = LogisticRegression(
    class_weight='balanced',  # Handle class imbalance
    max_iter=1000,           # Increase iterations for convergence
    random_state=42
)

# Initialize GridSearchCV with 5-fold cross-validation
lr_grid_search = GridSearchCV(
    estimator=lr_base,
    param_grid=lr_param_grid,
    scoring='roc_auc',       # Optimize for AUC-ROC
    cv=5,                    # 5-fold cross-validation
    verbose=1,
    n_jobs=-1                # Use all available cores
)

# Fit on scaled training data
lr_grid_search.fit(X_train_scaled, y_train)

# Get best model and parameters
lr_best = lr_grid_search.best_estimator_
lr_best_params = lr_grid_search.best_params_
lr_best_score = lr_grid_search.best_score_

print(f"\nBest Hyperparameters: {lr_best_params}")
print(f"Best CV AUC-ROC Score: {lr_best_score:.4f}")

# Make predictions on test set
lr_y_pred = lr_best.predict(X_test_scaled)
lr_y_pred_proba = lr_best.predict_proba(X_test_scaled)[:, 1]

print(f"\nPredictions made on test set ({len(y_test)} samples).")

### 5.2 Logistic Regression Coefficient Interpretation

Interpret the trained model by examining feature coefficients and calculating odds ratios.

In [ ]:
"""
Interpret Logistic Regression coefficients.

Coefficient interpretation:
- Positive coefficient: Increases log-odds of readmission
- Negative coefficient: Decreases log-odds of readmission

Odds Ratio (OR) = exp(coefficient):
- OR > 1: Feature increases odds of readmission
- OR < 1: Feature decreases odds of readmission
- OR = 1: No effect

Example: If age coefficient = 0.02, then OR = exp(0.02) = 1.02
Interpretation: Each additional year of age increases odds of readmission by 2%.
"""

print("Logistic Regression Coefficient Interpretation")
print("=" * 60)

# Extract coefficients
lr_coefficients = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': lr_best.coef_[0],
    'Odds Ratio': np.exp(lr_best.coef_[0])
})

# Sort by absolute coefficient value (importance)
lr_coefficients['Abs_Coefficient'] = lr_coefficients['Coefficient'].abs()
lr_coefficients = lr_coefficients.sort_values('Abs_Coefficient', ascending=False)

print("\nTop 10 Features by Coefficient Magnitude:")
display(lr_coefficients[['Feature', 'Coefficient', 'Odds Ratio']].head(10).round(4))

# Visualization
fig, ax = plt.subplots(figsize=(10, 8))

# Plot top 15 features
top_n = min(15, len(lr_coefficients))
top_features = lr_coefficients.head(top_n)

colors = ['red' if coef > 0 else 'blue' for coef in top_features['Coefficient']]
bars = ax.barh(top_features['Feature'], top_features['Coefficient'], color=colors, alpha=0.7)

ax.axvline(x=0, color='black', linestyle='-', linewidth=1)
ax.set_xlabel('Coefficient Value', fontsize=12)
ax.set_title('Logistic Regression Coefficients (Top 15 Features)', fontsize=14, fontweight='bold')
ax.invert_yaxis()  # Highest importance at top

# Add legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='red', alpha=0.7, label='Increases Readmission Risk'),
    Patch(facecolor='blue', alpha=0.7, label='Decreases Readmission Risk')
]
ax.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.savefig(os.path.join(DATA_PROCESSED_DIR, 'lr_coefficients.png'), dpi=150, bbox_inches='tight')
print(f"\nVisualization saved to {DATA_PROCESSED_DIR}/lr_coefficients.png")
plt.show()

# Clinical interpretation
print("\n*** Clinical Interpretation Guide: ***")
print("Positive coefficients (red) increase readmission risk.")
print("Negative coefficients (blue) decrease readmission risk.")
print("\nOdds Ratio interpretation:")
print("  - OR > 1.0: Feature increases odds of readmission")
print("  - OR < 1.0: Feature decreases odds of readmission")
print("  - Example: OR = 1.05 means 5% increase in odds per unit increase in feature")

### 5.3 Model 2: XGBoost (Custom Training)

Train an XGBoost classifier with custom hyperparameter tuning. XGBoost is a powerful gradient boosting algorithm that often outperforms traditional models.

In [ ]:
"""
Train XGBoost model with custom hyperparameter tuning.

Hyperparameters tuned:
- n_estimators: Number of boosting rounds (trees)
- max_depth: Maximum tree depth (controls complexity)
- learning_rate: Step size shrinkage (prevents overfitting)
- subsample: Fraction of samples used per tree
- colsample_bytree: Fraction of features used per tree
- scale_pos_weight: Handles class imbalance

XGBoost advantages:
- Handles non-linear relationships
- Robust to outliers and missing values
- Built-in feature importance
- Often achieves state-of-the-art performance
"""

if XGB_AVAILABLE:
    print("Training XGBoost Model")
    print("=" * 60)
    
    # Define parameter grid
    xgb_param_grid = {
        'n_estimators': [100, 200, 300],
        'max_depth': [3, 5, 7],
        'learning_rate': [0.01, 0.1, 0.2],
        'subsample': [0.8, 1.0],
        'colsample_bytree': [0.8, 1.0],
        'scale_pos_weight': [scale_pos_weight]  # Use calculated class weight
    }
    
    # Initialize XGBoost classifier
    xgb_base = xgb.XGBClassifier(
        objective='binary:logistic',
        random_state=42,
        use_label_encoder=False,
        eval_metric='auc'
    )
    
    # Initialize GridSearchCV
    xgb_grid_search = GridSearchCV(
        estimator=xgb_base,
        param_grid=xgb_param_grid,
        scoring='roc_auc',
        cv=3,  # 3-fold CV (XGBoost is slower)
        verbose=1,
        n_jobs=-1
    )
    
    # Fit on training data (XGBoost handles scaling internally, but we use scaled data for consistency)
    xgb_grid_search.fit(X_train_scaled, y_train)
    
    # Get best model and parameters
    xgb_best = xgb_grid_search.best_estimator_
    xgb_best_params = xgb_grid_search.best_params_
    xgb_best_score = xgb_grid_search.best_score_
    
    print(f"\nBest Hyperparameters: {xgb_best_params}")
    print(f"Best CV AUC-ROC Score: {xgb_best_score:.4f}")
    
    # Make predictions
    xgb_y_pred = xgb_best.predict(X_test_scaled)
    xgb_y_pred_proba = xgb_best.predict_proba(X_test_scaled)[:, 1]
    
    print(f"\nPredictions made on test set ({len(y_test)} samples).")
else:
    print("XGBoost not available. Skipping XGBoost training.")
    print("Install with: pip install xgboost")
    
    # Create placeholder variables to prevent errors
    xgb_best = None
    xgb_y_pred = None
    xgb_y_pred_proba = None
    xgb_best_params = {}

### 5.4 XGBoost Feature Importance

Extract and visualize feature importance from the trained XGBoost model.

In [ ]:
"""
Extract and visualize XGBoost feature importance.

Importance types:
- 'gain': Average gain (information gain) when feature is used in splits
- 'weight': Number of times feature appears in trees
- 'cover': Average coverage (number of samples affected) when feature is used

'gain' is generally the most interpretable as it measures contribution to model accuracy.
"""

if xgb_best is not None:
    print("XGBoost Feature Importance")
    print("=" * 60)
    
    # Extract feature importance (gain method)
    xgb_importance = pd.DataFrame({
        'Feature': X.columns,
        'Importance': xgb_best.feature_importances_
    })
    
    # Sort by importance
    xgb_importance = xgb_importance.sort_values('Importance', ascending=False)
    
    print("\nTop 10 Most Important Features:")
    display(xgb_importance.head(10).round(4))
    
    # Visualization
    fig, ax = plt.subplots(figsize=(10, 8))
    
    top_n = min(15, len(xgb_importance))
    top_features = xgb_importance.head(top_n)
    
    ax.barh(top_features['Feature'], top_features['Importance'], color='steelblue', alpha=0.7)
    ax.set_xlabel('Feature Importance (Gain)', fontsize=12)
    ax.set_title('XGBoost Feature Importance (Top 15)', fontsize=14, fontweight='bold')
    ax.invert_yaxis()
    
    plt.tight_layout()
    plt.savefig(os.path.join(DATA_PROCESSED_DIR, 'xgb_feature_importance.png'), dpi=150, bbox_inches='tight')
    print(f"\nVisualization saved to {DATA_PROCESSED_DIR}/xgb_feature_importance.png")
    plt.show()
    
    # Compare with LR coefficients
    print("\n*** Comparison with Logistic Regression: ***")
    print("Check if top features overlap between XGBoost and LR models.")
    print("Consistent top features across models increase confidence in their importance.")
else:
    print("XGBoost model not available. Skipping feature importance analysis.")

### 5.5 Comprehensive Model Evaluation

Evaluate both models using multiple metrics appropriate for imbalanced classification: Accuracy, Precision, Recall, F1-Score, and AUC-ROC.

In [ ]:
"""
Comprehensive evaluation of both models using multiple metrics.

Metrics explained:
- Accuracy: Overall correct predictions (can be misleading for imbalanced data)
- Precision: Of predicted readmissions, how many were actual readmissions? (TP / (TP + FP))
- Recall (Sensitivity): Of actual readmissions, how many did we catch? (TP / (TP + FN))
- F1-Score: Harmonic mean of Precision and Recall
- AUC-ROC: Area under Receiver Operating Characteristic curve (threshold-independent)

Clinical context:
- High Recall is critical: Missing a readmission risk (false negative) is costly
- Moderate Precision acceptable: False alarms (false positives) are less harmful than missed cases
"""

def evaluate_model(y_true, y_pred, y_pred_proba, model_name):
    """
    Calculate comprehensive evaluation metrics.
    
    Args:
        y_true: True labels
        y_pred: Predicted labels
        y_pred_proba: Predicted probabilities
        model_name: Name of the model
    
    Returns:
        dict: Dictionary of metric values
    """
    metrics = {
        'Model': model_name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'F1-Score': f1_score(y_true, y_pred),
        'AUC-ROC': roc_auc_score(y_true, y_pred_proba)
    }
    return metrics

# Evaluate Logistic Regression
lr_metrics = evaluate_model(y_test, lr_y_pred, lr_y_pred_proba, 'Logistic Regression')

# Evaluate XGBoost
if xgb_y_pred is not None:
    xgb_metrics = evaluate_model(y_test, xgb_y_pred, xgb_y_pred_proba, 'XGBoost')
else:
    xgb_metrics = None

# Display results
print("Model Evaluation Results")
print("=" * 70)

metrics_df = pd.DataFrame([lr_metrics])
if xgb_metrics:
    metrics_df = pd.concat([metrics_df, pd.DataFrame([xgb_metrics])], ignore_index=True)

# Format for display
display_df = metrics_df.copy()
for col in display_df.columns[1:]:
    display_df[col] = display_df[col].apply(lambda x: f"{x:.4f}")

display(display_df)

# Detailed interpretation
print("\n" + "=" * 70)
print("METRIC INTERPRETATION GUIDE")
print("=" * 70)
print("""
Accuracy: Overall correctness. Can be misleading if classes are imbalanced.
          Example: 85% accuracy means 85 out of 100 predictions are correct.

Precision: Positive predictive value. When model predicts readmission, how often is it right?
           Formula: TP / (TP + FP)
           Clinical: High precision = fewer false alarms, better resource allocation.

Recall: Sensitivity. Of all actual readmissions, how many did we identify?
        Formula: TP / (TP + FN)
        Clinical: HIGH RECALL IS CRITICAL. Low recall = missing high-risk patients.
        Example: Recall = 0.75 means we catch 75% of readmissions, miss 25%.

F1-Score: Balance between Precision and Recall. Useful single metric for imbalanced data.
          Formula: 2 × (Precision × Recall) / (Precision + Recall)

AUC-ROC: Threshold-independent measure of separability.
         0.5 = random guessing, 1.0 = perfect separation
         Clinical: AUC = 0.80 means model ranks random readmitted patient higher than 
                   random non-readmitted patient 80% of the time.
""")

# Highlight best model for each metric
if xgb_metrics:
    print("\n*** Best Performing Model by Metric: ***")
    for metric in ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']:
        best_model = max(metrics_df.to_dict('records'), key=lambda x: float(x[metric]))['Model']
        best_value = max(metrics_df[metric])
        print(f"  {metric}: {best_model} ({best_value:.4f})")

### 5.6 Confusion Matrix Analysis

Visualize confusion matrices for both models to understand error patterns (false positives vs false negatives).

In [ ]:
"""
Plot confusion matrices for both models.

Confusion Matrix components:
- True Negative (TN): Correctly predicted NOT readmitted
- False Positive (FP): Incorrectly predicted readmitted (Type I error)
- False Negative (FN): Incorrectly predicted NOT readmitted (Type II error) - MOST CRITICAL
- True Positive (TP): Correctly predicted readmitted

Clinical priority: Minimize False Negatives (missing high-risk patients)
even at the cost of some False Positives (unnecessary interventions).
"""

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix 1: Logistic Regression
cm_lr = confusion_matrix(y_test, lr_y_pred)
sns.heatmap(
    cm_lr,
    annot=True,
    fmt='d',
    cmap='Blues',
    cbar=False,
    ax=axes[0],
    annot_kws={'fontsize': 14}
)
axes[0].set_title('Logistic Regression Confusion Matrix', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Actual', fontsize=12)
axes[0].set_xlabel('Predicted', fontsize=12)
axes[0].set_xticklabels(['Not Readmitted', 'Readmitted'])
axes[0].set_yticklabels(['Not Readmitted', 'Readmitted'])

# Add TN, FP, FN, TP labels
tn_lr, fp_lr, fn_lr, tp_lr = cm_lr.ravel()
axes[0].text(0.5, -0.3, f'TN={tn_lr}', ha='center', va='center', transform=axes[0].transAxes, fontsize=10)
axes[0].text(1.5, -0.3, f'FP={fp_lr}', ha='center', va='center', transform=axes[0].transAxes, fontsize=10)
axes[0].text(0.5, 1.3, f'FN={fn_lr}', ha='center', va='center', transform=axes[0].transAxes, fontsize=10)
axes[0].text(1.5, 1.3, f'TP={tp_lr}', ha='center', va='center', transform=axes[0].transAxes, fontsize=10)

# Confusion Matrix 2: XGBoost
if xgb_y_pred is not None:
    cm_xgb = confusion_matrix(y_test, xgb_y_pred)
    sns.heatmap(
        cm_xgb,
        annot=True,
        fmt='d',
        cmap='Greens',
        cbar=False,
        ax=axes[1],
        annot_kws={'fontsize': 14}
    )
    axes[1].set_title('XGBoost Confusion Matrix', fontsize=14, fontweight='bold')
    axes[1].set_ylabel('Actual', fontsize=12)
    axes[1].set_xlabel('Predicted', fontsize=12)
    axes[1].set_xticklabels(['Not Readmitted', 'Readmitted'])
    axes[1].set_yticklabels(['Not Readmitted', 'Readmitted'])
    
    tn_xgb, fp_xgb, fn_xgb, tp_xgb = cm_xgb.ravel()
    axes[1].text(0.5, -0.3, f'TN={tn_xgb}', ha='center', va='center', transform=axes[1].transAxes, fontsize=10)
    axes[1].text(1.5, -0.3, f'FP={fp_xgb}', ha='center', va='center', transform=axes[1].transAxes, fontsize=10)
    axes[1].text(0.5, 1.3, f'FN={fn_xgb}', ha='center', va='center', transform=axes[1].transAxes, fontsize=10)
    axes[1].text(1.5, 1.3, f'TP={tp_xgb}', ha='center', va='center', transform=axes[1].transAxes, fontsize=10)
else:
    axes[1].text(0.5, 0.5, 'XGBoost not trained', ha='center', va='center', transform=axes[1].transAxes)
    axes[1].set_title('XGBoost Confusion Matrix', fontsize=14, fontweight='bold')

plt.suptitle('Confusion Matrix Comparison', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(DATA_PROCESSED_DIR, 'confusion_matrices.png'), dpi=150, bbox_inches='tight')
print(f"Visualization saved to {DATA_PROCESSED_DIR}/confusion_matrices.png")
plt.show()

# Interpretation
print("\n*** Confusion Matrix Interpretation: ***")
print(f"Logistic Regression: {fn_lr} false negatives (missed readmissions), {fp_lr} false positives (false alarms)")
if xgb_y_pred is not None:
    print(f"XGBoost: {fn_xgb} false negatives (missed readmissions), {fp_xgb} false positives (false alarms)")
print("\nClinical Priority: Minimize False Negatives (FN) - missing high-risk patients is costlier than false alarms.")

### 5.7 ROC Curve and Precision-Recall Curve

Visualize ROC curves and Precision-Recall curves for both models to assess performance across all thresholds.

In [ ]:
"""
Plot ROC curves and Precision-Recall curves.

ROC Curve:
- X-axis: False Positive Rate (FPR) = FP / (FP + TN)
- Y-axis: True Positive Rate (TPR/Recall) = TP / (TP + FN)
- AUC-ROC: Area under curve (higher is better, 0.5 = random)
- Threshold-independent measure of discriminative ability

Precision-Recall Curve:
- X-axis: Recall (TPR)
- Y-axis: Precision (PPV)
- More informative for imbalanced datasets than ROC
- Shows trade-off between catching cases (recall) and being accurate (precision)
"""

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
lr_fpr, lr_tpr, _ = roc_curve(y_test, lr_y_pred_proba)
lr_auc = roc_auc_score(y_test, lr_y_pred_proba)

axes[0].plot(lr_fpr, lr_tpr, color='blue', linewidth=2, label=f'LR (AUC = {lr_auc:.3f})')

if xgb_y_pred_proba is not None:
    xgb_fpr, xgb_tpr, _ = roc_curve(y_test, xgb_y_pred_proba)
    xgb_auc = roc_auc_score(y_test, xgb_y_pred_proba)
    axes[0].plot(xgb_fpr, xgb_tpr, color='green', linewidth=2, label=f'XGB (AUC = {xgb_auc:.3f})')

# Diagonal reference line (random classifier)
axes[0].plot([0, 1], [0, 1], color='gray', linestyle='--', linewidth=1, label='Random (AUC = 0.5)')

axes[0].set_xlabel('False Positive Rate', fontsize=12)
axes[0].set_ylabel('True Positive Rate (Recall)', fontsize=12)
axes[0].set_title('ROC Curve Comparison', fontsize=14, fontweight='bold')
axes[0].legend(loc='lower right')
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim([0, 1])
axes[0].set_ylim([0, 1.05])

# Precision-Recall Curve
lr_precision, lr_recall, _ = precision_recall_curve(y_test, lr_y_pred_proba)
lr_pr_auc = np.trapz(lr_precision, lr_recall)

axes[1].plot(lr_recall, lr_precision, color='blue', linewidth=2, label=f'LR (PR-AUC = {lr_pr_auc:.3f})')

if xgb_y_pred_proba is not None:
    xgb_precision, xgb_recall, _ = precision_recall_curve(y_test, xgb_y_pred_proba)
    xgb_pr_auc = np.trapz(xgb_precision, xgb_recall)
    axes[1].plot(xgb_recall, xgb_precision, color='green', linewidth=2, label=f'XGB (PR-AUC = {xgb_pr_auc:.3f})')

axes[1].set_xlabel('Recall', fontsize=12)
axes[1].set_ylabel('Precision', fontsize=12)
axes[1].set_title('Precision-Recall Curve Comparison', fontsize=14, fontweight='bold')
axes[1].legend(loc='lower left')
axes[1].grid(True, alpha=0.3)
axes[1].set_xlim([0, 1])
axes[1].set_ylim([0, 1.05])

plt.suptitle('Model Performance Curves', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(DATA_PROCESSED_DIR, 'roc_pr_curves.png'), dpi=150, bbox_inches='tight')
print(f"Visualization saved to {DATA_PROCESSED_DIR}/roc_pr_curves.png")
plt.show()

# Interpretation
print("\n*** ROC Curve Interpretation: ***")
print(f"Logistic Regression AUC-ROC: {lr_auc:.3f}")
if xgb_y_pred_proba is not None:
    print(f"XGBoost AUC-ROC: {xgb_auc:.3f}")
print("Higher AUC = better ability to distinguish readmitted from non-readmitted patients.")
print("AUC > 0.7 is acceptable, > 0.8 is good, > 0.9 is excellent.")

print("\n*** Precision-Recall Curve Interpretation: ***")
print("PR curves are more informative for imbalanced datasets.")
print("Higher PR-AUC indicates better precision-recall trade-off.")

<a id='section-6'></a>
## 6. Model Interpretation with SHAP

SHAP (SHapley Additive exPlanations) provides unified, theoretically-grounded explanations for model predictions. It shows:
- Global feature importance (which features matter most overall)
- Local explanations (why was THIS specific patient predicted high-risk?)
- Feature impact direction (does higher value increase or decrease risk?)

### 6.1 SHAP Summary Plot

Generate SHAP summary plots to visualize global feature importance and impact direction.

In [ ]:
"""
Generate SHAP summary plots for XGBoost model.

Summary plot shows:
- Features ranked by importance (vertical axis)
- SHAP value on horizontal axis (impact on prediction)
- Color indicates feature value (red = high, blue = low)
- Each dot is one patient

Interpretation:
- Right side (positive SHAP): Increases readmission risk
- Left side (negative SHAP): Decreases readmission risk
- Red dots on right: High feature values increase risk
- Blue dots on left: Low feature values decrease risk
"""

if SHAP_AVAILABLE and xgb_best is not None:
    print("Generating SHAP Explanations")
    print("=" * 60)
    
    # Create SHAP explainer for tree-based models
    explainer = shap.TreeExplainer(xgb_best)
    
    # Calculate SHAP values for test set
    print("Calculating SHAP values (this may take a minute)...")
    shap_values = explainer.shap_values(X_test_scaled)
    
    print(f"SHAP values calculated for {len(X_test)} test samples.")
    
    # Summary plot (beeswarm)
    plt.figure(figsize=(12, 10))
    shap.summary_plot(
        shap_values,
        X_test,
        feature_names=X.columns.tolist(),
        plot_type='dot',
        show=False,
        color_bar=True,
        max_display=15
    )
    plt.title('SHAP Summary Plot - XGBoost Model', fontsize=14, fontweight='bold', pad=20)
    plt.savefig(os.path.join(DATA_PROCESSED_DIR, 'shap_summary_beeswarm.png'), dpi=150, bbox_inches='tight')
    print(f"Beeswarm summary plot saved to {DATA_PROCESSED_DIR}/shap_summary_beeswarm.png")
    plt.show()
    
    # Summary plot (bar)
    plt.figure(figsize=(10, 8))
    shap.summary_plot(
        shap_values,
        X_test,
        feature_names=X.columns.tolist(),
        plot_type='bar',
        show=False,
        max_display=15
    )
    plt.title('SHAP Feature Importance - XGBoost Model', fontsize=14, fontweight='bold', pad=20)
    plt.savefig(os.path.join(DATA_PROCESSED_DIR, 'shap_summary_bar.png'), dpi=150, bbox_inches='tight')
    print(f"Bar summary plot saved to {DATA_PROCESSED_DIR}/shap_summary_bar.png")
    plt.show()
    
    # Clinical interpretation guide
    print("\n" + "=" * 70)
    print("SHAP SUMMARY PLOT INTERPRETATION GUIDE")
    print("=" * 70)
    print("""
How to read the beeswarm plot:

1. Vertical axis: Features ranked by importance (top = most important)

2. Horizontal axis: SHAP value
   - Positive (right): Increases predicted readmission risk
   - Negative (left): Decreases predicted readmission risk

3. Color: Feature value
   - Red: High feature value
   - Blue: Low feature value

4. Each dot: One patient from the test set

Example interpretations:
- If 'comorbidity_count' red dots are on the right: Patients with MANY comorbidities 
  have HIGHER readmission risk
- If 'bmi' blue dots are on the left: Patients with LOW BMI have LOWER readmission risk
- If 'hba1c_imputed' red dots spread both ways: HbA1c has complex, non-linear relationship

Clinical Insight: Focus on top 3-5 features for actionable interventions.
Example: If 'prior_admissions' is top feature, implement enhanced discharge planning 
         for patients with multiple prior admissions.
""")
else:
    print("SHAP or XGBoost not available. Skipping SHAP analysis.")
    print("Install with: pip install shap")

### 6.2 SHAP Dependence Plots

Examine how specific features affect predictions, including potential non-linear relationships and interactions.

In [ ]:
"""
Generate SHAP dependence plots for top features.

Dependence plot shows:
- X-axis: Feature value
- Y-axis: SHAP value (impact on prediction)
- Color: Interaction feature (automatically selected)

This reveals:
- Non-linear relationships (curved patterns)
- Threshold effects (sudden changes at certain values)
- Feature interactions (color patterns)
"""

if SHAP_AVAILABLE and xgb_best is not None:
    print("Generating SHAP Dependence Plots")
    print("=" * 60)
    
    # Get top 3 features from SHAP summary
    shap_summary = np.abs(shap_values).mean(axis=0)
    top_indices = np.argsort(shap_summary)[::-1][:3]
    top_features = [X.columns[i] for i in top_indices]
    
    print(f"Top 3 features for dependence plots: {top_features}")
    
    # Create dependence plots for top 3 features
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    for idx, feature in enumerate(top_features):
        shap.dependence_plot(
            feature,
            shap_values,
            X_test,
            feature_names=X.columns.tolist(),
            ax=axes[idx],
            show=False
        )
        axes[idx].set_title(f'SHAP Dependence: {feature}', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(os.path.join(DATA_PROCESSED_DIR, 'shap_dependence_plots.png'), dpi=150, bbox_inches='tight')
    print(f"Dependence plots saved to {DATA_PROCESSED_DIR}/shap_dependence_plots.png")
    plt.show()
    
    print("\n*** Dependence Plot Interpretation: ***")
    print("Look for:")
    print("  - Upward slope: Higher values increase risk")
    print("  - Downward slope: Higher values decrease risk")
    print("  - Curved patterns: Non-linear relationships")
    print("  - Color gradients: Interactions with other features")
else:
    print("SHAP or XGBoost not available. Skipping dependence plots.")

### 6.3 SHAP Force Plots (Individual Predictions)

Explain individual patient predictions using SHAP force plots. This shows why a specific patient was predicted high-risk or low-risk.

In [ ]:
"""
Generate SHAP force plots for individual patient predictions.

Force plot shows:
- Base value: Average model prediction across all patients
- Red arrows: Features pushing prediction HIGHER (increase risk)
- Blue arrows: Features pushing prediction LOWER (decrease risk)
- Arrow length: Magnitude of feature's impact
- Final value: Model's predicted probability for this patient

Use case: Explain to clinicians WHY a specific patient is high-risk.
Example: "Patient A has 85% readmission risk primarily due to:
          - 5 prior admissions (+25%)
          - 8 comorbidities (+20%)
          - Age 92 (+15%)"
"""

if SHAP_AVAILABLE and xgb_best is not None:
    print("Generating SHAP Force Plots for Individual Predictions")
    print("=" * 60)
    
    # Select example patients: one high-risk, one low-risk
    pred_probs = xgb_y_pred_proba
    
    # Find high-risk patient (predicted probability > 0.7)
    high_risk_idx = np.argmax(pred_probs)
    
    # Find low-risk patient (predicted probability < 0.3)
    low_risk_idx = np.argmin(pred_probs)
    
    print(f"Selected patients for explanation:")
    print(f"  High-risk: Patient index {high_risk_idx}, predicted probability = {pred_probs[high_risk_idx]:.3f}")
    print(f"  Low-risk:  Patient index {low_risk_idx}, predicted probability = {pred_probs[low_risk_idx]:.3f}")
    
    # Create force plots
    fig, axes = plt.subplots(2, 1, figsize=(12, 8))
    
    # High-risk patient
    shap.force_plot(
        explainer.expected_value,
        shap_values[high_risk_idx],
        X_test.iloc[high_risk_idx],
        feature_names=X.columns.tolist(),
        matplotlib=True,
        show=False,
        text_rotation=0,
        ax=axes[0]
    )
    axes[0].set_title(f'High-Risk Patient (Predicted: {pred_probs[high_risk_idx]:.1%})', fontsize=14, fontweight='bold', pad=10)
    
    # Low-risk patient
    shap.force_plot(
        explainer.expected_value,
        shap_values[low_risk_idx],
        X_test.iloc[low_risk_idx],
        feature_names=X.columns.tolist(),
        matplotlib=True,
        show=False,
        text_rotation=0,
        ax=axes[1]
    )
    axes[1].set_title(f'Low-Risk Patient (Predicted: {pred_probs[low_risk_idx]:.1%})', fontsize=14, fontweight='bold', pad=10)
    
    plt.tight_layout()
    plt.savefig(os.path.join(DATA_PROCESSED_DIR, 'shap_force_plots.png'), dpi=150, bbox_inches='tight')
    print(f"Force plots saved to {DATA_PROCESSED_DIR}/shap_force_plots.png")
    plt.show()
    
    print("\n*** Force Plot Interpretation Guide: ***")
    print("Base value: Average prediction across all patients (starting point)")
    print("Red bars (right): Features increasing readmission risk")
    print("Blue bars (left): Features decreasing readmission risk")
    print("Bar length: Magnitude of feature's contribution")
    print("Final value (right edge): Model's predicted probability for this patient")
    print("\nClinical Use: Show these plots to care teams to explain individual patient risk factors.")
else:
    print("SHAP or XGBoost not available. Skipping force plots.")

<a id='section-7'></a>
## 7. Conclusion and Next Steps

### 7.1 Summary of Achievements

This notebook has completed all requirements for Progress Review 1:

✅ **Data Collection & Preparation:**
- Defined eICU-CRD v2.0 feature extraction mapping
- Implemented CSV-based data loading with fallback to synthetic data
- Handled missing HbA1c with imputation + indicator variable
- Encoded categorical variables appropriately
- Applied feature scaling and stratified train-test split
- Addressed class imbalance with weighted loss

✅ **Exploratory Data Analysis:**
- Missing data analysis with heatmap and bar charts
- Target variable distribution with imbalance assessment
- Feature distributions with KDE plots
- Age category analysis with readmission rates
- Correlation heatmap with hierarchical clustering
- Primary diagnosis distribution analysis

✅ **Model Training & Evaluation:**
- Logistic Regression baseline with hyperparameter tuning
- XGBoost with custom hyperparameter optimization
- Comprehensive metrics: Accuracy, Precision, Recall, F1, AUC-ROC
- Confusion matrix analysis with TN/TP/FN/FP breakdown
- ROC and Precision-Recall curves
- Feature importance (LR coefficients + XGBoost gain)

✅ **Model Interpretation:**
- SHAP summary plots (beeswarm and bar)
- SHAP dependence plots for top features
- SHAP force plots for individual predictions
- Clinical interpretation guides throughout

### 7.2 Key Findings (To Be Populated After Execution)

*Complete this section after running the notebook with actual data:*

1. **Most predictive features:** [List top 3-5 features from SHAP analysis]
2. **Model performance:** [Compare LR vs XGBoost AUC-ROC and Recall]
3. **Clinical implications:** [What interventions could reduce readmissions?]
4. **Data quality insights:** [HbA1c availability, class imbalance ratio, etc.]

### 7.3 Limitations

1. **Synthetic data:** Current results use synthetic data. Real eICU-CRD validation needed.
2. **Temporal validation:** No time-based split (future work: train on earlier years, test on later).
3. **External validation:** Model needs testing on different hospital systems.
4. **Clinical utility:** Requires prospective study to assess impact on readmission rates.

### 7.4 Next Steps for Progress Review 2

1. **Obtain eICU-CRD access:** Complete PhysioNet credentialing process
2. **Run on real data:** Execute full pipeline with actual eICU-CRD v2.0
3. **Advanced modeling:**
   - Try ensemble methods (stacking LR + XGBoost)
   - Implement SMOTE for oversampling minority class
   - Calibrate probabilities (Platt scaling, isotonic regression)
4. **Threshold optimization:** Select optimal threshold based on clinical costs
5. **Deployment preparation:**
   - Save trained models (pickle/joblib)
   - Create prediction API (Flask/FastAPI)
   - Build simple web interface for clinicians
6. **Ethical considerations:** Document fairness analysis across demographic groups

### 7.5 Experiment Tracking Log

| Date | Model | Hyperparameters | AUC-ROC | Recall | Notes |
|------|-------|-----------------|---------|--------|-------|
| TBD | Logistic Regression | C=__, solver=__ | __.__ | __.__ | Baseline model |
| TBD | XGBoost | n_est=__, depth=__, lr=__ | __.__ | __.__ | Best performing |

### 7.6 References

1. Pollard TJ, et al. The eICU Collaborative Research Database, a freely available multi-center database for critical care research. Scientific Data. 2018;5:180178.
2. Lundberg SM, Lee S-I. A Unified Approach to Interpreting Model Predictions. NeurIPS. 2017.
3. Chen T, Guestrin C. XGBoost: A Scalable Tree Boosting System. KDD. 2016.
4. Van Walraven C, et al. A multivariable prediction model for hospital readmission within thirty days of discharge. JAMA Internal Medicine. 2015.